# Load dependencies

In [ ]:
import math, matplotlib.pyplot as plt, operator, torch
from functools import partial

In [ ]:
torch.manual_seed(42)
torch.set_printoptions(precision=3, linewidth=140, sci_mode=False)

# Create data

In [ ]:
from torch.distributions.multivariate_normal import MultivariateNormal
from torch import tensor


n_clusters=6
n_samples =250
synthetic_data_mean = 30
synthetic_data_std = 50
center_cov_mat = torch.diag(tensor([5.,5.]))
# center_cov_mat = torch.tensor([[10.,2.],[2.,10.]])

centroids = (torch.rand(n_clusters, 2) - 0.5) * synthetic_data_std + synthetic_data_mean
print(centroids)

def sample(m):
  return MultivariateNormal(m, center_cov_mat).sample((n_samples,))

slices = [sample(c) for c in centroids]
data = torch.cat(slices)
print(data.shape)

# Plot data

In [ ]:
def plot_data(centroids, data, n_samples, ax=None):
    if ax is None: _,ax = plt.subplots()
    for i, centroid in enumerate(centroids):
        samples = data[i*n_samples:(i+1)*n_samples]
        ax.scatter(samples[:,0], samples[:,1], s=1)
        ax.plot(*centroid, markersize=10, marker="x", color='k', mew=5)
        ax.plot(*centroid, markersize=5, marker="x", color='m', mew=2)

In [ ]:
plot_data(centroids, data, n_samples)

# Mean shift

Most people that have come across clustering algorithms have learnt about **k-means**.
Mean shift clustering is a newer and less well-known approach, but it has some important advantages:

- It doesn't require selecting the number of clusters in advance, but instead just requires a **bandwidth** to be specified, which can be easily chosen automatically
- It can handle clusters of any shape, whereas k-means (without using special extensions) requires that clusters be roughly ball shaped.

The algorithm is as follows:

- For each data point x in the sample X, find the distance between that point x and every other point in X
- Create weights for each point in X by using the **Gaussian kernel** of that point's distance to x
  - This weighting approach penalizes points further away from x
  - The rate at which the weights fall to zero is determined by the **bandwidth**, which is the standard deviation of the Gaussian
- Update x as the weighted average of all other points in X, weighted based on the previous step

This will iteratively push points that are close together even closer until they are next to each other.

In [ ]:
midp = data.mean(0)
print(midp)
plot_data([midp]*6, data, n_samples)

## Kernels

### Gaussian kernel

In [ ]:
def gaussian(d, bw): return torch.exp(-0.5*((d/bw))**2) / (bw*math.sqrt(2*math.pi))

def plot_func(f):
    x = torch.linspace(0,10,100)
    plt.plot(x, f(x))

plot_func(partial(gaussian, bw=2.5))

### Linear kernel

In [ ]:
def tri(d, i): return (-d+i).clamp_min(0)/i
plot_func(partial(tri, i=8))

## Inefficient implementation

In [ ]:
X = data.clone()
x = data[0]
print(f"initial x: {x}")
print(f"x.shape,X.shape: {x.shape,X.shape}")
print(f"(x-X)[:8]:\n{(x-X)[:8]}")
dist = ((x-X)**2).sum(1).sqrt() # Broadcast
print(f"dist[:8]:\n{dist[:8]}")
weight = gaussian(dist, 2.5)
print(f"weight[:8]:\n{weight[:8]}")

print(f"weight.shape,X.shape: {weight.shape,X.shape}") # axis augmentation needed
new_x = (weight[:,None]*X).sum(0)/weight.sum()
print(f"new_x: {new_x}")

In [ ]:
def one_update(X):
    for i, x in enumerate(X):
        dist = torch.sqrt(((x-X)**2).sum(1))
        weight = gaussian(dist, 2.5)
        # weight = tri(dist, 8)
        X[i] = (weight[:,None]*X).sum(0)/weight.sum()

def meanshift(data):
    X = data.clone()
    for it in range(5): one_update(X)
    return X

%time X=meanshift(data)

In [ ]:
plot_data(centroids+2, X, n_samples)

# Animation

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def do_one(d):
    if d: one_update(X)
    ax.clear()
    # fix xlim and ylim
    ax.set_xlim(0,80)
    ax.set_ylim(0,80)
    plot_data(centroids+2, X, n_samples, ax=ax)

# create your own animation
X = data.clone()
fig,ax = plt.subplots()

ani = FuncAnimation(fig, do_one, frames=5, interval=500, repeat=False)
plt.close()
HTML(ani.to_jshtml())

# GPU batched algorithm

To truly accelerate the algorithm, we need to be performing updates on a batch of points per iteration, instead of just one as we were doing.

In [ ]:
bs=5
X = data.clone()
x = X[:bs]
print(x.shape,X.shape)
print(X[None].shape, x[:,None].shape) # Needed to use boradcast

In [ ]:
def dist_b(a,b): return (((a[None]-b[:,None])**2).sum(2)).sqrt()

dist_pt = dist_b(X, x)
print(dist_pt.shape)
print(dist_pt)

weight = gaussian(dist_pt, 2)
print(weight.shape)
print(weight)

## Calculate weighted sum
- Broadcast
- Eisnum
- Pytorch matmul

In [ ]:
div = weight.sum(1, keepdim=True)

new_x_1 = (weight[...,None]*X[None]).sum(1) / div
new_x_2 = torch.einsum('ij,jk->ik', weight, X) / div
new_x_3 = weight@X / div
print(new_x_1)
print(new_x_2)
print(new_x_3)

In [ ]:
def meanshift(data, bs):
    n = len(data)
    X = data.clone()
    for it in range(5):
        for i in range(0, n, bs):
            s = slice(i, min(i+bs,n))
            weight = gaussian(dist_b(X, X[s]), 2.5)
            # weight = tri(dist_b(X, X[s]), 8)
            div = weight.sum(1, keepdim=True)
            X[s] = weight@X/div
    return X

Although each iteration still has to launch a new cuda kernel, there are now fewer iterations, and the acceleration from updating a batch of points more than makes up for it.

In [ ]:
data = data.cuda()
X = meanshift(data, bs=512).cpu()

In [ ]:
%timeit -n 5 _=meanshift(data, 512).cpu()

about 230x faster than the previous version


In [ ]:
plot_data(centroids+2, X, n_samples)